# n-gram 言語モデルを作ってパープレキシティを測り、文を生成する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_corpus_statistics/ngram_lm.ipynb)

NLP クックブック 07-10 ／ コーパスの統計と可視化

古典的な n-gram 言語モデルを学習して、文の「それらしさ」を数値化したい。文も生成してみたい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, re, urllib.request, zipfile
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
text = aozora("https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip")
sents = [[w.surface for w in tagger(s)] for s in re.split(r"(?<=。)|\n", text) if s and s.strip()]
train, test = sents[:-300], sents[-300:]
print(len(train), len(test))

8769 300


In [3]:
from nltk.lm import KneserNeyInterpolated, Laplace, Vocabulary
from nltk.lm.preprocessing import pad_both_ends, padded_everygram_pipeline
from nltk.util import ngrams

n = 3
models = {}
for name, cls in [("Laplace", Laplace), ("Kneser-Ney", KneserNeyInterpolated)]:
    data, words = padded_everygram_pipeline(n, train)
    lm = cls(n, vocabulary=Vocabulary(words, unk_cutoff=2))  # 1 回しか出ない語は <UNK> として学習
    lm.fit(data)
    models[name] = lm

def perplexity(lm, sentences):
    grams = [g for s in sentences for g in ngrams(pad_both_ends(lm.vocab.lookup(s), n=n), n)]
    return lm.perplexity(grams)

for name, lm in models.items():
    print(f"{name:11} テスト PPL = {perplexity(lm, test[:100]):.1f}")

Laplace     テスト PPL = 1144.6
Kneser-Ney  テスト PPL = 85.6


In [4]:
lm = models["Kneser-Ney"]
for s in ["吾輩は猫である。", "猫は吾輩である。", "である猫吾輩は。"]:
    print(f"{perplexity(lm, [[w.surface for w in tagger(s)]]):10.1f}  {s}")

for seed in range(3):
    out = lm.generate(30, text_seed=["<s>", "<s>"], random_seed=seed)
    print("".join(w for w in out if w not in ("<s>", "</s>")))

       3.6  吾輩は猫である。
      13.6  猫は吾輩である。
     294.4  である猫吾輩は。
早々鈴木君に対して授業以外に大得意である。
「本物たあ何だか心細い事があって、暗黒の<UNK>をしないわ、――羽より軽いものだ。
警察の<UNK>が案出されたものである。


- 未知語があると確率 0 (PPL = ∞) になるので、低頻度語を `<UNK>` に置き換えて学習する (`unk_cutoff`)。
- スムージングで未知の n-gram の確率を 0 にしないのが肝。加算 (Laplace) より Kneser-Ney のほうがずっと良い (PPL が低い)。
- 本格的に使うなら KenLM (高速・省メモリ)。ニューラル言語モデルのパープレキシティは Transformers の章を参照。